## 1. Azure + vector store setup

In [17]:
from pathlib import Path
import os, json, re, time, math
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from openai import AzureOpenAI
from IPython.display import display, Markdown

ROOT = Path(".")
POLICY_DIR = ROOT / "data" / "healthcare_policies"
ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

load_dotenv(".env", override=True)

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_MODEL")
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")

assert AZURE_OPENAI_API_KEY, "Missing AZURE_OPENAI_API_KEY in .env"
assert AZURE_OPENAI_ENDPOINT, "Missing AZURE_OPENAI_ENDPOINT in .env"
assert CHAT_DEPLOYMENT, "Missing AZURE_OPENAI_MODEL in .env"

# --- Azure OpenAI client (one client handles both chat + embeddings) ---
client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

env_df = pd.DataFrame([
    {"Component":"Chat / Generation", "Azure deployment":CHAT_DEPLOYMENT, "Purpose":"Generate grounded answers / rerank"},
    {"Component":"Embeddings", "Azure deployment":EMBEDDING_DEPLOYMENT, "Purpose":"Convert text and queries into vectors"}
])
display(env_df)

import chromadb

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

records_by_strategy = {
    s: load_jsonl(ARTIFACT_DIR / f"policy_chunks_{s}.jsonl")
    for s in ["fixed", "sentence", "recursive", "section"]
}

corpus_df = pd.DataFrame([
    {"Strategy":s, "Chunks":len(rows), "Avg chars":round(np.mean([len(x["text"]) for x in rows]), 1)}
    for s, rows in records_by_strategy.items()
])
display(corpus_df)

model = CHAT_DEPLOYMENT
embedding_model = EMBEDDING_DEPLOYMENT

,Component,Azure deployment,Purpose
0,Chat / Generation,gpt-4.1,Generate grounded answers / rerank
1,Embeddings,text-embedding-3-small,Convert text and queries into vectors


,Strategy,Chunks,Avg chars
0,fixed,16,581.8
1,sentence,17,482.2
2,recursive,17,539.5
3,section,34,214.7


2. Load PDFs, Recursive Chunking using Langchain and store the vectors in langchain Chroma DB

In [18]:
# --- LangChain imports for PDF loading + recursive chunking + vector store ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma

# ============================================================
# 0) Config
# ============================================================
CHUNK_SIZE     = 1000
CHUNK_OVERLAP  = 150
RAW_EXTS       = {".pdf"}                   # raw policies are PDF files
VECTOR_DB_PATH = str(ARTIFACT_DIR / "chroma_policy_db copy")

# --- Embeddings object (used for vector storage) ---
lc_embeddings = AzureOpenAIEmbeddings(
    azure_deployment=EMBEDDING_DEPLOYMENT,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

# --- Recursive character text splitter ---
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    is_separator_regex=False,
)

# ============================================================
# 1) Load PDFs from POLICY_DIR (page-by-page)
# ============================================================
page_docs = []   # LangChain Document objects, one per PDF page
for fp in sorted(POLICY_DIR.rglob("*")):
    if not fp.is_file() or fp.suffix.lower() not in RAW_EXTS:
        continue
    try:
        pages = PyPDFLoader(str(fp)).load()      # one Document per page
    except Exception as e:
        print(f"⚠️  Skipping {fp.name} — could not read PDF ({e})")
        continue

    for pg in pages:
        if not pg.page_content.strip():          # skip blank/scanned-image pages
            continue
        # Normalize metadata onto our standard schema
        pg.metadata = {
            "doc_id":         fp.stem,
            "source_file":    fp.name,
            "title":          fp.stem,
            "plan_type":      "",
            "policy_domain":  "",
            "effective_date": "",
            "page":           pg.metadata.get("page", ""),   # 0-based page index from PyPDF
            "section":        "",
            "chunk_strategy": "recursive",
        }
        page_docs.append(pg)

assert page_docs, (
    f"No readable text found in PDFs under {POLICY_DIR.resolve()}. "
    f"If these are scanned/image PDFs, they need OCR before text extraction."
)
n_files = len({d.metadata['source_file'] for d in page_docs})
print(f"✅ Loaded {len(page_docs)} page(s) from {n_files} PDF file(s) in {POLICY_DIR.resolve()}")

# ============================================================
# 2) Chunk with LangChain (split_documents preserves page metadata)
# ============================================================
recursive_docs = recursive_splitter.split_documents(page_docs)
assert recursive_docs, "Splitter produced 0 chunks — extracted pages may be empty."

# Assign a stable chunk_id per doc
chunk_ids, seen = [], {}
for d in recursive_docs:
    did = d.metadata["doc_id"]
    idx = seen.get(did, 0)
    seen[did] = idx + 1
    cid = f"{did}_rec_{idx}"
    d.metadata["chunk_id"] = cid
    chunk_ids.append(cid)

print(f"✅ Created {len(recursive_docs)} chunks from {len(page_docs)} page(s).")

# ============================================================
# 3) Embed + store via LangChain Chroma (fresh rebuild each run)
# ============================================================
try:
    Chroma(collection_name="policy_recursive_lc_1",
           persist_directory=VECTOR_DB_PATH,
           embedding_function=lc_embeddings).delete_collection()
except Exception:
    pass

vectorstore = Chroma.from_documents(
    documents=recursive_docs,
    embedding=lc_embeddings,
    collection_name="policy_recursive_lc_1",
    persist_directory=VECTOR_DB_PATH,
    collection_metadata={"hnsw:space": "cosine"},
    ids=chunk_ids,
)

stored = vectorstore._collection.count()
assert stored > 0, "Chroma stored 0 vectors — embedding step returned nothing."

collections = {"recursive": vectorstore}

vector_store_df = pd.DataFrame([
    {"Collection": v._collection.name, "Strategy": s, "Vectors Stored": v._collection.count()}
    for s, v in collections.items()
])
display(vector_store_df)

✅ Loaded 5 page(s) from 5 PDF file(s) in C:\Users\AP45804\Documents\HPP AI Learning Sessions\data\healthcare_policies
✅ Created 11 chunks from 5 page(s).


,Collection,Strategy,Vectors Stored
0,policy_recursive_lc_1,recursive,11


LLM Reranking function & Langchain Retrieval Function

In [19]:
def rerank_with_llm(question, initial_df):
    candidates = []
    for _, row in initial_df.iterrows():
        candidates.append({
            "rank":     int(row["Rank"]),
            "chunk_id": f"C{int(row['Rank'])}",
            "document": row["Document"],
            "section":  row["Section"],
            "text":     row["Text"],          # was row["Retrieved Text"]
        })

    prompt = f"""
You are reranking retrieved healthcare policy chunks for a user question.

QUESTION:
{question}

CANDIDATES:
{json.dumps(candidates, indent=2)}

Score every candidate from 0 to 100 for how directly it contains evidence needed to answer the question.
Return ONLY a JSON array in this format:
[
  {{"chunk_id":"C1","relevance_score":95,"reason":"..."}}
]
Do not omit any candidate.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"You are a precise enterprise search reranker."},
            {"role":"user","content":prompt}
        ],
        temperature=0
    )

    text = response.choices[0].message.content.strip()
    text = re.sub(r"^```json\s*|\s*```$", "", text, flags=re.I|re.S)
    scores = json.loads(text)
    score_map = {x["chunk_id"]: x for x in scores}

    reranked = initial_df.copy()
    reranked["Candidate"] = [f"C{x}" for x in reranked["Rank"]]
    reranked["Rerank Score ↑"] = reranked["Candidate"].map(
        lambda x: score_map.get(x, {}).get("relevance_score", 0)
    )
    reranked["Rerank Reason"] = reranked["Candidate"].map(
        lambda x: score_map.get(x, {}).get("reason", "")
    )

    reranked = reranked.sort_values(
        ["Rerank Score ↑", "Similarity"], ascending=[False, False]   # was "Cosine Similarity ↑"
    ).reset_index(drop=True)

    reranked["Rank After Rerank"] = np.arange(1, len(reranked)+1)
    reranked["Rank Movement"] = reranked["Rank"] - reranked["Rank After Rerank"]
    return reranked


# --- Initial retrieval via LangChain (replaces semantic_search + search_to_df) ---
def langchain_search_to_df(question, vectorstore, top_k=6):
    hits = vectorstore.similarity_search_with_relevance_scores(question, k=top_k)
    rows = []
    for rank, (doc, score) in enumerate(hits, start=1):
        rows.append({
            "Rank":       rank,
            "Similarity": round(score, 4),
            "Document":   doc.metadata.get("title", doc.metadata.get("doc_id", "")),
            "Section":    doc.metadata.get("section", ""),
            "Text":       doc.page_content,
        })
    return pd.DataFrame(rows)


rerank_question = "What clinical documentation is required when requesting continued physical therapy?"

# collection is now "recursive" (the one built in the previous blocks), not "section"
initial = langchain_search_to_df(
    rerank_question, collections["recursive"], top_k=6
)

display(
    initial[
        ["Rank", "Similarity", "Document", "Section", "Text"]
    ].rename(columns={"Rank":"Vector Rank"})
)

,Vector Rank,Similarity,Document,Section,Text
0,1,0.6305,04_Rehabilitation_Therapy_Policy_2026,,"status, progress toward measurable goals, trea..."
1,2,0.5741,04_Rehabilitation_Therapy_Policy_2026,,Synthetic training document - no real member d...
2,3,0.4867,01_Gold_PPO_2026_Benefits_Authorization,,SECTION 3: PHYSICAL THERAPY\nThe first 10 phys...
3,4,0.4766,02_Silver_HMO_2026_Benefits_Authorization,,treatment plan.\nSECTION 4: SPECIALIST REFERRA...
4,5,0.4306,02_Silver_HMO_2026_Benefits_Authorization,,Synthetic training document - no real member d...
5,6,0.4164,03_Advanced_Imaging_Utilization_Management_2026,,Synthetic training document - no real member d...


LLM Based Reranking Example output

In [20]:
reranked = rerank_with_llm(rerank_question, initial)

display(
    reranked[
        ["Rank","Rank After Rerank","Rank Movement","Similarity",
         "Rerank Score ↑","Document","Section","Rerank Reason"]
    ].rename(columns={"Rank":"Vector Rank"})
)

,Vector Rank,Rank After Rerank,Rank Movement,Similarity,Rerank Score ↑,Document,Section,Rerank Reason
0,1,1,0,0.6305,95,04_Rehabilitation_Therapy_Policy_2026,,Directly states that clinical documentation mu...
1,2,2,0,0.5741,90,04_Rehabilitation_Therapy_Policy_2026,,Specifies that requests for continued physical...
2,3,3,0,0.4867,85,01_Gold_PPO_2026_Benefits_Authorization,,States that authorization requests for continu...
3,5,4,1,0.4306,80,02_Silver_HMO_2026_Benefits_Authorization,,States that continued therapy requests must in...
4,4,5,-1,0.4766,30,02_Silver_HMO_2026_Benefits_Authorization,,Mentions treatment plan and specialist referra...
5,6,6,0,0.4164,0,03_Advanced_Imaging_Utilization_Management_2026,,"Pertains to advanced imaging, not physical the..."


Testing the RAG Pipeline with 2 Examples

In [21]:
def build_context(df, max_chars=6000):
    """Assemble retrieved chunks into a cit-able context string."""
    blocks, used = [], 0
    for i, row in df.iterrows():
        tag = f"[{i+1}]"
        header = f"{tag} {row['Document']}" + (f" — {row['Section']}" if row['Section'] else "")
        snippet = row["Text"].strip()
        piece = f"{header}\n{snippet}"
        if used + len(piece) > max_chars:      # stay within a safe context budget
            break
        blocks.append(piece)
        used += len(piece)
    return "\n\n".join(blocks)


def answer_question(question, top_k=6, rerank=True, keep_top=4):
    # 1) Retrieve
    retrieved = langchain_search_to_df(question, collections["recursive"], top_k=top_k)
    if retrieved.empty:
        return "No relevant policy content was found for this question.", retrieved

    # 2) Optional LLM rerank, then keep the strongest few
    if rerank:
        ranked = rerank_with_llm(question, retrieved).head(keep_top).reset_index(drop=True)
    else:
        ranked = retrieved.head(keep_top).reset_index(drop=True)

    # 3) Build grounded context + generate
    context = build_context(ranked)
    prompt = f"""Answer the user's question using ONLY the policy context below.
Cite the sources you use with their bracket numbers, e.g. [1], [2].
If the answer is not in the context, say so explicitly — do not invent details.

QUESTION:
{question}

POLICY CONTEXT:
{context}

Answer:"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"system","content":"You are a precise healthcare policy assistant. Ground every claim in the provided context and cite sources."},
            {"role":"user","content":prompt}
        ],
        temperature=0
    )
    answer = response.choices[0].message.content.strip()
    return answer, ranked


# ============================================================
# Test Prompts
# ============================================================
user_prompt_1 = "What clinical documentation is required when requesting continued physical therapy?"

answer, sources = answer_question(user_prompt_1, top_k=6, rerank=True, keep_top=4)

display(Markdown(f"### Question 1\n{user_prompt_1}\n\n### Answer\n{answer}"))

print("\nSources used:")
display(
    sources[["Document","Section","Similarity"]]
    .reset_index(drop=True)
    .rename(columns={"Similarity":"Cosine Similarity ↑"})
)

user_prompt_2 = "Compare the chiropractic benefit between the Gold PPO & Silver HMO Plan"

answer, sources = answer_question(user_prompt_2, top_k=6, rerank=True, keep_top=4)

display(Markdown(f"### Question 2\n{user_prompt_2}\n\n### Answer\n{answer}"))

print("\nSources used:")
display(
    sources[["Document","Section","Similarity"]]
    .reset_index(drop=True)
    .rename(columns={"Similarity":"Cosine Similarity ↑"})
)

### Question 1
What clinical documentation is required when requesting continued physical therapy?

### Answer
When requesting continued physical therapy, clinical documentation must include:

- Diagnosis
- Baseline functional status
- Progress toward measurable goals
- Treatment frequency
- Expected duration
- Clinical progress and a treatment plan (for Silver HMO members)
- Documentation supporting the services billed, available for utilization review or post-payment audit

Additionally, continued therapy must demonstrate measurable functional improvement or a documented maintenance need as allowed by the member's benefit plan. Services that are solely custodial are not covered [1], [2], [3], [4].


Sources used:


,Document,Section,Cosine Similarity ↑
0,04_Rehabilitation_Therapy_Policy_2026,,0.6305
1,04_Rehabilitation_Therapy_Policy_2026,,0.5741
2,01_Gold_PPO_2026_Benefits_Authorization,,0.4867
3,02_Silver_HMO_2026_Benefits_Authorization,,0.4306


### Question 2
Compare the chiropractic benefit between the Gold PPO & Silver HMO Plan

### Answer
The Gold PPO plan covers up to 20 medically necessary chiropractic visits per benefit year. Visits beyond this annual limit are not covered unless an approved exception applies. No primary-care referral is required to access chiropractic services under the Gold PPO plan, but services may be subject to prior authorization requirements if listed on the prior-authorization schedule [1], [4].

The Silver HMO plan covers up to 12 medically necessary chiropractic visits per benefit year. A referral from the member's assigned primary-care provider is required before a routine chiropractic visit. Authorization or referral approval does not guarantee claim payment; eligibility, network status, benefit limits, and coding requirements are evaluated at claim adjudication [2], [3].

In summary:
- Gold PPO: 20 visits/year, no referral required, possible prior authorization [1], [4].
- Silver HMO: 12 visits/year, referral required, subject to claim adjudication [2], [3].


Sources used:


,Document,Section,Cosine Similarity ↑
0,01_Gold_PPO_2026_Benefits_Authorization,,0.5523
1,02_Silver_HMO_2026_Benefits_Authorization,,0.4693
2,02_Silver_HMO_2026_Benefits_Authorization,,0.5125
3,01_Gold_PPO_2026_Benefits_Authorization,,0.4880
